In [2]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

index_name = "logs"

mapping = {
    "mappings": {
        "properties": {
            "timestamp": {"type": "date"},
            "level": {"type": "keyword"},
            "service": {"type": "keyword"},
            "message": {"type": "text"}
        }
    }
}

if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name, body=mapping)

print("index logs créé")

index logs créé


In [5]:
from datetime import datetime

logs = [
    {"timestamp": datetime.now(), "level": "INFO", "service": "auth", "message": "user login"},
    {"timestamp": datetime.now(), "level": "ERROR", "service": "payment", "message": "payment failed"},
    {"timestamp": datetime.now(), "level": "WARNING", "service": "auth", "message": "Password retry"},
    {"timestamp": datetime.now(), "level": "ERROR", "service": "order", "message": "Order not found"},
    {"timestamp": datetime.now(), "level": "INFO", "service": "payment", "message": "Payment success"},
]

for i, log in enumerate(logs):
    es.index(index="logs", id=i, document=log)

print("Logs insérés")

Logs insérés


In [6]:
res = es.search(
    index="logs",
    query={
        "match": {"level": "ERROR"}
    }
)

for hit in res["hits"]["hits"]:
    print(hit["_source"])

{'timestamp': '2026-04-01T19:15:04.919100', 'level': 'ERROR', 'service': 'payment', 'message': 'payment failed'}
{'timestamp': '2026-04-01T19:15:04.919103', 'level': 'ERROR', 'service': 'order', 'message': 'Order not found'}


In [7]:
res = es.search(
    index="logs",
    query={
        "bool": {
            "must": [
                {"match": {"level": "ERROR"}},
                {"match": {"service": "payment"}}
            ]
        }
    }
)

print(res["hits"]["hits"])

[{'_index': 'logs', '_id': '1', '_score': 1.7509375, '_source': {'timestamp': '2026-04-01T19:15:04.919100', 'level': 'ERROR', 'service': 'payment', 'message': 'payment failed'}}]


# Etape 5 __ Aggregations

In [8]:
res = es.search(
    index="logs",
    size=0,
    aggs={
        "errors_by_service": {
            "terms": {"field": "service"}
        }
    }
)

print(res["aggregations"])

{'errors_by_service': {'doc_count_error_upper_bound': 0, 'sum_other_doc_count': 0, 'buckets': [{'key': 'auth', 'doc_count': 2}, {'key': 'payment', 'doc_count': 2}, {'key': 'order', 'doc_count': 1}]}}
